# MATH 5010 Computer Lab — Section 10  
## Sampling Methods: Monte Carlo  
### Full Solutions Included

This lab accompanies **Section 10: Sampling Methods — Monte Carlo**.

We will use Python to study:

1. Monte Carlo simulation and the Law of Large Numbers  
2. Estimating $\pi$ by random points  
3. Buffon's needle experiment  
4. Monte Carlo integration  
5. Error rate and confidence intervals for Monte Carlo estimators  
6. Random number generation  
7. Inverse-transform sampling  
8. Box--Muller sampling for normal random variables  
9. Rejection sampling  
10. Importance sampling  
11. Sampling-importance-resampling  
12. Random search and simulated annealing  
13. Practice problems with complete solutions

The main message is that Monte Carlo methods convert hard sums, probabilities, and integrals into expectations that can be approximated by simulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate, optimize
from math import sqrt, pi, exp, log, sin, cos

rng = np.random.default_rng(5010)

pd.set_option("display.precision", 6)
print("Packages loaded.")

## 1. Monte Carlo Principle

Suppose we want to compute

\[
\theta = E[g(X)]
\]

where $X$ has density $p(x)$. Then

\[
\theta
=
\int g(x)p(x)\,dx.
\]

Monte Carlo estimates this expectation by drawing

\[
X_1,\ldots,X_N\sim p(x)
\]

and using

\[
\hat\theta_N
=
\frac1N\sum_{i=1}^N g(X_i).
\]

By the Law of Large Numbers,

\[
\hat\theta_N \to \theta
\]

as $N\to\infty$.

In [ ]:
# Example: estimate E[X^2] for X ~ Uniform(0,1)
# True value = ∫_0^1 x^2 dx = 1/3

N_values = np.array([10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000])
true_value = 1/3

rows = []
for N in N_values:
    X = rng.uniform(0, 1, size=N)
    estimate = np.mean(X**2)
    rows.append([N, estimate, abs(estimate - true_value)])

pd.DataFrame(rows, columns=["N", "Monte Carlo estimate", "Absolute error"])

In [ ]:
# Running average illustration
N = 50_000
X = rng.uniform(0, 1, size=N)
running_estimate = np.cumsum(X**2) / np.arange(1, N+1)

plt.figure(figsize=(7, 4))
plt.plot(running_estimate, label="running Monte Carlo estimate")
plt.axhline(true_value, linestyle="--", label="true value 1/3")
plt.xlabel("Number of samples")
plt.ylabel(r"Estimate of $E[X^2]$")
plt.title("Monte Carlo Estimate Converges by LLN")
plt.legend()
plt.show()

### Full Solution

Here

\[
X\sim U(0,1),\qquad g(X)=X^2.
\]

The exact expectation is

\[
E[X^2]=\int_0^1 x^2\,dx=\frac13.
\]

The Monte Carlo estimator is

\[
\hat\theta_N=\frac1N\sum_{i=1}^N X_i^2.
\]

By the Law of Large Numbers,

\[
\hat\theta_N \xrightarrow{P} \frac13.
\]

## 2. Estimating $\pi$ by Random Points

Sample random points uniformly in the square

\[
[0,1]\times[0,1].
\]

The quarter circle

\[
x^2+y^2\le 1
\]

has area $\pi/4$. Therefore,

\[
P(X^2+Y^2\le 1)=\frac{\pi}{4}.
\]

So

\[
\pi \approx 4\cdot \frac{\#\{(X_i,Y_i):X_i^2+Y_i^2\le1\}}{N}.
\]

In [ ]:
def estimate_pi(N, seed=None):
    local_rng = np.random.default_rng(seed)
    X = local_rng.uniform(0, 1, size=N)
    Y = local_rng.uniform(0, 1, size=N)
    inside = (X**2 + Y**2 <= 1)
    return 4 * inside.mean()

N_values = [100, 1_000, 10_000, 100_000, 1_000_000]
rows = []
for N in N_values:
    est = estimate_pi(N, seed=5010 + N)
    rows.append([N, est, abs(est - np.pi)])

pd.DataFrame(rows, columns=["N", "pi estimate", "absolute error"])

In [ ]:
# Plot random points for a small sample
N = 4_000
X = rng.uniform(0, 1, size=N)
Y = rng.uniform(0, 1, size=N)
inside = X**2 + Y**2 <= 1

theta = np.linspace(0, np.pi/2, 300)

plt.figure(figsize=(6, 6))
plt.scatter(X[inside], Y[inside], s=4, alpha=0.5, label="inside quarter circle")
plt.scatter(X[~inside], Y[~inside], s=4, alpha=0.5, label="outside")
plt.plot(np.cos(theta), np.sin(theta), linewidth=2, label="quarter circle")
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Monte Carlo Estimation of pi")
plt.legend()
plt.show()

### Full Solution

The point $(X,Y)$ is uniformly distributed on the unit square, whose area is $1$. The event

\[
X^2+Y^2\le1
\]

is the quarter unit disk, whose area is $\pi/4$. Therefore,

\[
P(X^2+Y^2\le1)=\frac{\pi}{4}.
\]

Let

\[
I_i=1\{X_i^2+Y_i^2\le1\}.
\]

Then

\[
E[I_i]=\frac{\pi}{4}.
\]

The Monte Carlo estimate is

\[
\hat\pi=4\bar I.
\]

## 3. Speed of Monte Carlo Convergence

Monte Carlo error usually decreases at rate

\[
O\left(\frac1{\sqrt{N}}\right).
\]

If

\[
\hat\theta_N=\frac1N\sum_{i=1}^N g(X_i),
\]

then approximately

\[
\operatorname{SE}(\hat\theta_N)
=
\frac{\operatorname{SD}(g(X))}{\sqrt N}.
\]

This is slow but dimension-independent in a useful way.

In [ ]:
# Empirical error scaling for pi estimation
reps = 300
N_values = np.array([100, 300, 1_000, 3_000, 10_000, 30_000])

rows = []
for N in N_values:
    estimates = np.array([estimate_pi(N, seed=10_000 + r + N) for r in range(reps)])
    rmse = np.sqrt(np.mean((estimates - np.pi)**2))
    rows.append([N, rmse, rmse*np.sqrt(N)])

df = pd.DataFrame(rows, columns=["N", "RMSE", "RMSE * sqrt(N)"])
display(df)

plt.figure(figsize=(7, 4))
plt.loglog(df["N"], df["RMSE"], marker="o", label="empirical RMSE")
plt.loglog(df["N"], df["RMSE"].iloc[0]*np.sqrt(df["N"].iloc[0]/df["N"]), linestyle="--", label=r"reference $1/\sqrt{N}$")
plt.xlabel("N")
plt.ylabel("RMSE")
plt.title("Monte Carlo Error Rate")
plt.legend()
plt.show()

### Full Solution

For the $\pi$ estimator,

\[
\hat\pi=4\bar I,
\]

where

\[
I_i\sim \mathrm{Bernoulli}(\pi/4).
\]

Thus

\[
\operatorname{Var}(\hat\pi)
=
16\frac{(\pi/4)(1-\pi/4)}{N}.
\]

Therefore,

\[
\operatorname{SE}(\hat\pi)
=
4\sqrt{\frac{(\pi/4)(1-\pi/4)}{N}},
\]

which has order $1/\sqrt{N}$.

## 4. Buffon's Needle Problem

Suppose parallel lines are distance $D$ apart and a needle of length $L\le D$ is dropped randomly.

The probability that the needle crosses a line is

\[
P(\text{cross})=\frac{2L}{\pi D}.
\]

Thus,

\[
\pi=\frac{2L}{DP(\text{cross})}.
\]

If $k$ out of $n$ trials cross a line, then

\[
\hat\pi=\frac{2Ln}{Dk}.
\]

In [ ]:
def buffon_simulation(n, L=1.0, D=1.0, seed=None):
    local_rng = np.random.default_rng(seed)
    # Distance from needle center to nearest line is Uniform(0, D/2)
    y = local_rng.uniform(0, D/2, size=n)
    # Angle with line direction; use theta in [0, pi/2] by symmetry
    theta = local_rng.uniform(0, np.pi/2, size=n)
    crosses = y <= (L/2) * np.sin(theta)
    k = crosses.sum()
    pi_hat = np.inf if k == 0 else 2*L*n/(D*k)
    return pi_hat, k/n

N_values = [1_000, 10_000, 100_000, 1_000_000]
rows = []
for N in N_values:
    pi_hat, p_cross_hat = buffon_simulation(N, L=1, D=1, seed=2026+N)
    rows.append([N, p_cross_hat, pi_hat, abs(pi_hat - np.pi)])

pd.DataFrame(rows, columns=["n", "estimated crossing probability", "pi estimate", "absolute error"])

### Full Solution

Let $Y$ be the distance from the needle center to the nearest line. Then

\[
Y\sim U(0,D/2).
\]

Let $\Theta$ be the angle between the needle and the line direction. By symmetry,

\[
\Theta\sim U(0,\pi/2).
\]

The needle crosses a line if

\[
Y\le \frac L2\sin\Theta.
\]

The crossing probability is

\[
P(\text{cross})
=
E\left[P\left(Y\le \frac L2\sin\Theta\mid\Theta\right)\right]
=
E\left[\frac{L\sin\Theta}{D}\right].
\]

Since

\[
E[\sin\Theta]=\frac{2}{\pi},
\]

we get

\[
P(\text{cross})=\frac{2L}{\pi D}.
\]

Solving gives

\[
\pi=\frac{2L}{D P(\text{cross})}.
\]

## 5. Monte Carlo Integration

Suppose we want to evaluate

\[
I=\int_0^1 e^{-x^3}\,dx.
\]

Let $U\sim U(0,1)$. Then

\[
I=E[e^{-U^3}].
\]

So the Monte Carlo estimator is

\[
\hat I=\frac1N\sum_{i=1}^N e^{-U_i^3}.
\]

In [ ]:
def mc_integral_exp_x3(N, seed=None):
    local_rng = np.random.default_rng(seed)
    U = local_rng.uniform(0, 1, size=N)
    values = np.exp(-U**3)
    estimate = values.mean()
    se = values.std(ddof=1) / np.sqrt(N)
    return estimate, se

true_integral, _ = integrate.quad(lambda x: np.exp(-x**3), 0, 1)

N_values = [100, 1_000, 10_000, 100_000]
rows = []
for N in N_values:
    est, se = mc_integral_exp_x3(N, seed=123+N)
    rows.append([N, est, se, est - true_integral])

pd.DataFrame(rows, columns=["N", "MC estimate", "estimated SE", "error"])

In [ ]:
# One Monte Carlo run with CI
N = 20_000
est, se = mc_integral_exp_x3(N, seed=99)
ci_low = est - 1.96*se
ci_high = est + 1.96*se

print("True integral:", true_integral)
print("MC estimate:", est)
print("Estimated SE:", se)
print("Approximate 95% Monte Carlo CI:", (ci_low, ci_high))

### Full Solution

Since $U\sim U(0,1)$ has density $1$ on $[0,1]$,

\[
E[e^{-U^3}]
=
\int_0^1 e^{-u^3}\,du.
\]

Thus,

\[
\hat I_N
=
\frac1N\sum_{i=1}^N e^{-U_i^3}
\]

is unbiased:

\[
E[\hat I_N]=I.
\]

Its standard error is estimated by

\[
\widehat{\operatorname{SE}}(\hat I_N)
=
\frac{s}{\sqrt N},
\]

where $s$ is the sample standard deviation of $e^{-U_i^3}$.

## 6. Monte Carlo for Multidimensional Integration

Consider

\[
I_d=\int_{[0,1]^d} \exp\left(-\sum_{j=1}^d x_j^2\right)\,dx.
\]

If

\[
U=(U_1,\ldots,U_d)\sim U([0,1]^d),
\]

then

\[
I_d=E\left[\exp\left(-\sum_{j=1}^d U_j^2\right)\right].
\]

In [ ]:
def mc_integral_dimension(d, N=100_000, seed=None):
    local_rng = np.random.default_rng(seed)
    U = local_rng.uniform(0, 1, size=(N, d))
    values = np.exp(-np.sum(U**2, axis=1))
    return values.mean(), values.std(ddof=1)/np.sqrt(N)

rows = []
for d in [1, 2, 5, 10, 20, 50]:
    est, se = mc_integral_dimension(d, N=100_000, seed=500+d)
    rows.append([d, est, se])

pd.DataFrame(rows, columns=["dimension d", "MC estimate", "estimated SE"])

### Full Solution

Because $U$ is uniform on the unit cube, its density is $1$ on $[0,1]^d$. Therefore,

\[
E\left[\exp\left(-\sum_{j=1}^d U_j^2\right)\right]
=
\int_{[0,1]^d}\exp\left(-\sum_{j=1}^d x_j^2\right)\,dx.
\]

The Monte Carlo estimator is

\[
\hat I_d
=
\frac1N\sum_{i=1}^N
\exp\left(-\sum_{j=1}^d U_{ij}^2\right).
\]

This works in any dimension, though variance and sampling efficiency may still become challenging in high dimensions.

## 7. Random Number Generation and Inverse Transform Sampling

If $F$ is a CDF and $U\sim U(0,1)$, then

\[
X=F^{-1}(U)
\]

has CDF $F$.

This is called the **inverse-transform method**.

### Example: Exponential Distribution

For $X\sim \mathrm{Exponential}(\lambda)$,

\[
F(x)=1-e^{-\lambda x}.
\]

Set $U=F(X)$:

\[
U=1-e^{-\lambda X}.
\]

Solving for $X$ gives

\[
X=-\frac{1}{\lambda}\log(1-U).
\]

Since $1-U\sim U(0,1)$, we often use

\[
X=-\frac{1}{\lambda}\log U.
\]

In [ ]:
N = 100_000
lam = 2.0

U = rng.uniform(0, 1, size=N)
X = -np.log(U) / lam

grid = np.linspace(0, np.quantile(X, 0.995), 400)
pdf = lam * np.exp(-lam * grid)

plt.figure(figsize=(7, 4))
plt.hist(X, bins=70, density=True, alpha=0.7, label="inverse-transform samples")
plt.plot(grid, pdf, label="Exponential density")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Inverse Transform Sampling: Exponential")
plt.legend()
plt.show()

print("Sample mean:", X.mean())
print("Theoretical mean:", 1/lam)

### Full Solution

Let

\[
X=-\frac{1}{\lambda}\log U.
\]

For $x\ge0$,

\[
P(X\le x)
=
P\left(-\frac1\lambda\log U\le x\right)
=
P(\log U\ge -\lambda x)
=
P(U\ge e^{-\lambda x})
=
1-e^{-\lambda x}.
\]

Thus $X\sim \mathrm{Exponential}(\lambda)$.

## 8. Inverse Transform Sampling for a Discrete Distribution

Suppose

\[
P(X=0)=0.2,\quad P(X=1)=0.5,\quad P(X=2)=0.3.
\]

The CDF jumps at $0,1,2$. If $U\sim U(0,1)$, define

\[
X=
\begin{cases}
0, & 0<U\le0.2,\\
1, & 0.2<U\le0.7,\\
2, & 0.7<U\le1.
\end{cases}
\]

In [ ]:
N = 100_000
U = rng.uniform(0, 1, size=N)

X = np.where(U <= 0.2, 0, np.where(U <= 0.7, 1, 2))

empirical = pd.Series(X).value_counts(normalize=True).sort_index()
theory = pd.Series([0.2, 0.5, 0.3], index=[0, 1, 2])

pd.DataFrame({"Empirical probability": empirical, "Theory": theory})

### Full Solution

The inverse CDF for a discrete distribution maps intervals of length equal to the probability mass onto the corresponding value.

Here:

\[
F(0)=0.2,\qquad F(1)=0.7,\qquad F(2)=1.
\]

Therefore,

\[
F^{-1}(u)=
\begin{cases}
0, & 0<u\le0.2,\\
1, & 0.2<u\le0.7,\\
2, & 0.7<u\le1.
\end{cases}
\]

The simulated frequencies should be close to the target probabilities.

## 9. Box--Muller Method for Normal Random Variables

Let

\[
U_1,U_2\sim U(0,1)
\]

independently. Define

\[
R=\sqrt{-2\log U_1},
\qquad
\Theta=2\pi U_2.
\]

Then

\[
Z_1=R\cos\Theta,
\qquad
Z_2=R\sin\Theta
\]

are independent standard normal random variables.

In [ ]:
N = 100_000

U1 = rng.uniform(0, 1, size=N)
U2 = rng.uniform(0, 1, size=N)

R = np.sqrt(-2*np.log(U1))
Theta = 2*np.pi*U2

Z1 = R*np.cos(Theta)
Z2 = R*np.sin(Theta)

print("Mean Z1:", Z1.mean())
print("Variance Z1:", Z1.var(ddof=0))
print("Mean Z2:", Z2.mean())
print("Variance Z2:", Z2.var(ddof=0))
print("Correlation between Z1 and Z2:", np.corrcoef(Z1, Z2)[0,1])

In [ ]:
grid = np.linspace(-4, 4, 400)

plt.figure(figsize=(7, 4))
plt.hist(Z1, bins=70, density=True, alpha=0.7, label="Box-Muller Z1")
plt.plot(grid, stats.norm.pdf(grid), label="N(0,1) density")
plt.xlabel("z")
plt.ylabel("Density")
plt.title("Box-Muller Normal Samples")
plt.legend()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(Z1[:3000], Z2[:3000], s=5, alpha=0.4)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("Z1")
plt.ylabel("Z2")
plt.title("Box-Muller Produces Independent Standard Normals")
plt.show()

### Full Solution

The joint density of two independent standard normals is

\[
f(z_1,z_2)=\frac1{2\pi}\exp\left(-\frac{z_1^2+z_2^2}{2}\right).
\]

In polar coordinates,

\[
z_1=r\cos\theta,\qquad z_2=r\sin\theta,
\]

with Jacobian $r$, so

\[
f_{R,\Theta}(r,\theta)
=
\frac1{2\pi}e^{-r^2/2}r.
\]

Thus $\Theta\sim U(0,2\pi)$ and

\[
F_R(r)=1-e^{-r^2/2}.
\]

Using inverse transform,

\[
R=\sqrt{-2\log U_1},
\qquad
\Theta=2\pi U_2.
\]

Then $Z_1=R\cos\Theta$ and $Z_2=R\sin\Theta$ are independent standard normals.

## 10. Rejection Sampling

Suppose we want to sample from a target density $p(z)$.

Choose a proposal density $q(z)$ and a constant $k$ such that

\[
p(z)\le kq(z)
\]

for all $z$.

Algorithm:

1. Sample $Z\sim q$.
2. Sample $U\sim U(0,1)$.
3. Accept $Z$ if

\[
U\le \frac{p(Z)}{kq(Z)}.
\]

Otherwise reject and try again.

### Example: Sample from Beta$(2,5)$ using Uniform$(0,1)$ proposal

In [ ]:
# Target: Beta(2,5)
# Proposal: Uniform(0,1), q(x)=1
# k = max p(x), found numerically

a, b = 2, 5

def target_pdf(x):
    return stats.beta.pdf(x, a, b)

# Find maximum of beta density on (0,1)
res = optimize.minimize_scalar(lambda x: -target_pdf(x), bounds=(0, 1), method="bounded")
k = -res.fun
mode = res.x

print("Approximate mode:", mode)
print("k = max p(x):", k)

def rejection_beta(N):
    accepted = []
    total_proposals = 0
    while len(accepted) < N:
        m = max(1000, 2*(N - len(accepted)))
        z = rng.uniform(0, 1, size=m)
        u = rng.uniform(0, 1, size=m)
        accept = u <= target_pdf(z) / k
        accepted.extend(z[accept].tolist())
        total_proposals += m
    return np.array(accepted[:N]), total_proposals

N = 50_000
samples_beta, total_proposals = rejection_beta(N)
accept_rate = N / total_proposals

print("Acceptance rate:", accept_rate)
print("Expected acceptance rate roughly:", 1/k)

In [ ]:
grid = np.linspace(0, 1, 400)

plt.figure(figsize=(7, 4))
plt.hist(samples_beta, bins=70, density=True, alpha=0.7, label="Rejection samples")
plt.plot(grid, target_pdf(grid), label="Beta(2,5) target density")
plt.plot(grid, k*np.ones_like(grid), linestyle="--", label="k q(x)")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Rejection Sampling from Beta(2,5)")
plt.legend()
plt.show()

### Full Solution

With proposal

\[
q(x)=1,\qquad 0<x<1,
\]

we need

\[
p(x)\le kq(x)=k.
\]

Thus $k$ is the maximum of the target density on $[0,1]$.

The acceptance probability is

\[
P(\text{accept})
=
\int q(x)\frac{p(x)}{kq(x)}\,dx
=
\frac1k\int p(x)\,dx
=
\frac1k.
\]

Accepted samples have density proportional to

\[
q(x)\frac{p(x)}{kq(x)}
=
\frac{p(x)}{k},
\]

which normalizes to $p(x)$.

## 11. Rejection Sampling for an Unnormalized Density

Often we know only an unnormalized target density

\[
\tilde p(x)=e^{-x^4/4},
\qquad -\infty<x<\infty.
\]

We can still use rejection sampling if we can compare with a proposal.

Here we use a normal proposal $q(x)=N(0,1)$ and choose a numerical constant $k$ such that

\[
\tilde p(x)\le kq(x)
\]

on a large practical range. This produces samples from a density proportional to $\tilde p(x)$.

In [ ]:
# Unnormalized target
def p_tilde(x):
    return np.exp(-x**4/4)

def q_pdf(x):
    return stats.norm.pdf(x, loc=0, scale=1)

# Numerically find max of p_tilde/q on a grid
grid = np.linspace(-6, 6, 20001)
ratio = p_tilde(grid) / q_pdf(grid)
k = ratio.max()
x_max = grid[np.argmax(ratio)]

print("Approximate k:", k)
print("Max ratio at x:", x_max)

def rejection_unnormalized(N):
    accepted = []
    total = 0
    while len(accepted) < N:
        m = max(2000, 3*(N - len(accepted)))
        z = rng.normal(0, 1, size=m)
        u = rng.uniform(0, 1, size=m)
        accept_prob = p_tilde(z) / (k * q_pdf(z))
        accept = u <= accept_prob
        accepted.extend(z[accept].tolist())
        total += m
    return np.array(accepted[:N]), total

N = 30_000
samples, total = rejection_unnormalized(N)
print("Acceptance rate:", N/total)

In [ ]:
# Normalize target numerically for plotting
Z_const, _ = integrate.quad(lambda x: np.exp(-x**4/4), -np.inf, np.inf)

x_plot = np.linspace(-3, 3, 500)
target_norm = p_tilde(x_plot) / Z_const

plt.figure(figsize=(7, 4))
plt.hist(samples, bins=80, density=True, alpha=0.7, label="samples")
plt.plot(x_plot, target_norm, label="normalized target")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Rejection Sampling with Unnormalized Target")
plt.legend()
plt.show()

### Full Solution

Rejection sampling only needs the ratio

\[
\frac{p(z)}{kq(z)}.
\]

If

\[
p(z)=\frac{\tilde p(z)}{Z_p},
\]

where $Z_p$ is unknown, we can often work with $\tilde p$ by choosing a constant $k$ for the unnormalized comparison.

The accepted sample has density proportional to

\[
q(z)\cdot \frac{\tilde p(z)}{kq(z)}
=
\frac{\tilde p(z)}{k}.
\]

After normalization, this is the target density proportional to $\tilde p(z)$.

## 12. Importance Sampling

Suppose we want

\[
I=E_p[f(Z)]=\int f(z)p(z)\,dz.
\]

If we sample from a different distribution $q(z)$, then

\[
I
=
\int f(z)\frac{p(z)}{q(z)}q(z)\,dz
=
E_q\left[f(Z)\frac{p(Z)}{q(Z)}\right].
\]

The weight is

\[
w(z)=\frac{p(z)}{q(z)}.
\]

### Example: Estimate a rare normal tail

Estimate

\[
P(Z>5)
\]

for $Z\sim N(0,1)$.

In [ ]:
true_tail = 1 - stats.norm.cdf(5)

def crude_tail_mc(N, seed=None):
    local_rng = np.random.default_rng(seed)
    Z = local_rng.normal(0, 1, size=N)
    values = (Z > 5).astype(float)
    return values.mean(), values.std(ddof=1)/np.sqrt(N)

def importance_tail_mc(N, proposal_mean=5, seed=None):
    local_rng = np.random.default_rng(seed)
    X = local_rng.normal(proposal_mean, 1, size=N)  # q = N(5,1)
    p_density = stats.norm.pdf(X, 0, 1)
    q_density = stats.norm.pdf(X, proposal_mean, 1)
    weights = p_density / q_density
    values = (X > 5).astype(float) * weights
    return values.mean(), values.std(ddof=1)/np.sqrt(N)

N = 100_000
crude_est, crude_se = crude_tail_mc(N, seed=1)
is_est, is_se = importance_tail_mc(N, proposal_mean=5, seed=1)

pd.DataFrame({
    "Method": ["Crude Monte Carlo", "Importance sampling"],
    "Estimate": [crude_est, is_est],
    "Estimated SE": [crude_se, is_se],
    "True value": [true_tail, true_tail]
})

### Full Solution

The probability can be written as

\[
P(Z>5)=E_p[1\{Z>5\}],
\]

where $p$ is the standard normal density.

Choose proposal

\[
q=N(5,1).
\]

Then

\[
P(Z>5)
=
E_q\left[
1\{X>5\}\frac{p(X)}{q(X)}
\right].
\]

This works much better than crude Monte Carlo because the proposal puts many more samples in the rare-event region $x>5$.

## 13. Self-Normalized Importance Sampling

Sometimes the target density is known only up to a constant:

\[
p(z)=\frac{\tilde p(z)}{Z_p}.
\]

Then

\[
E_p[f(Z)]
\approx
\sum_{i=1}^N W_i f(Z_i),
\]

where

\[
W_i=\frac{\tilde p(Z_i)/q(Z_i)}{\sum_{m=1}^N \tilde p(Z_m)/q(Z_m)}.
\]

This is called **self-normalized importance sampling**.

In [ ]:
# Target proportional to exp(-x^4/4), proposal q=N(0,1)
# Estimate E_p[X^2]

N = 200_000
X = rng.normal(0, 1, size=N)

raw_weights = p_tilde(X) / q_pdf(X)
norm_weights = raw_weights / raw_weights.sum()

estimate_x2 = np.sum(norm_weights * X**2)

# Numerical true value
Z_const, _ = integrate.quad(lambda x: np.exp(-x**4/4), -np.inf, np.inf)
num_x2, _ = integrate.quad(lambda x: x**2 * np.exp(-x**4/4), -np.inf, np.inf)
true_x2 = num_x2 / Z_const

print("Self-normalized IS estimate E[X^2]:", estimate_x2)
print("Numerical true value:", true_x2)

# Effective sample size
ess = 1 / np.sum(norm_weights**2)
print("Effective sample size:", ess)
print("N:", N)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(norm_weights, bins=80)
plt.xlabel("normalized weight")
plt.ylabel("frequency")
plt.title("Distribution of Self-Normalized Importance Weights")
plt.show()

### Full Solution

The expectation is

\[
E_p[f(Z)]
=
\frac{\int f(z)\tilde p(z)\,dz}{\int \tilde p(z)\,dz}.
\]

Using samples $Z_i\sim q$,

\[
\int f(z)\tilde p(z)\,dz
=
\int f(z)\frac{\tilde p(z)}{q(z)}q(z)\,dz
\approx
\frac1N\sum_{i=1}^N f(Z_i)\frac{\tilde p(Z_i)}{q(Z_i)}.
\]

Similarly,

\[
\int \tilde p(z)\,dz
\approx
\frac1N\sum_{i=1}^N \frac{\tilde p(Z_i)}{q(Z_i)}.
\]

Taking the ratio gives

\[
\hat E_p[f]
=
\frac{\sum_i f(Z_i)\tilde p(Z_i)/q(Z_i)}
{\sum_i \tilde p(Z_i)/q(Z_i)}
=
\sum_i W_i f(Z_i).
\]

## 14. Sampling-Importance-Resampling

Sampling-importance-resampling, or SIR, uses importance weights to convert proposal samples into approximate target samples.

Algorithm:

1. Draw $Z_1,\ldots,Z_L\sim q(z)$.
2. Compute weights

\[
w_i\propto \frac{\tilde p(Z_i)}{q(Z_i)}.
\]

3. Normalize weights.
4. Resample from $\{Z_1,\ldots,Z_L\}$ with probabilities $w_i$.

This avoids finding the rejection-sampling constant $k$.

In [ ]:
L = 100_000
M = 20_000

proposal_samples = rng.normal(0, 1, size=L)
raw_w = p_tilde(proposal_samples) / q_pdf(proposal_samples)
w = raw_w / raw_w.sum()

indices = rng.choice(np.arange(L), size=M, replace=True, p=w)
sir_samples = proposal_samples[indices]

x_plot = np.linspace(-3, 3, 500)
target_norm = p_tilde(x_plot) / Z_const

plt.figure(figsize=(7, 4))
plt.hist(sir_samples, bins=80, density=True, alpha=0.7, label="SIR samples")
plt.plot(x_plot, target_norm, label="target density")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Sampling-Importance-Resampling")
plt.legend()
plt.show()

print("Mean of SIR samples:", sir_samples.mean())
print("Variance of SIR samples:", sir_samples.var(ddof=0))
print("Target E[X^2] numerically:", true_x2)

### Full Solution

After sampling from $q$, points in regions where the target is large relative to $q$ receive larger weights. Resampling with probabilities

\[
w_i=\frac{\tilde p(Z_i)/q(Z_i)}
{\sum_m \tilde p(Z_m)/q(Z_m)}
\]

produces a new sample that approximately follows the target distribution.

Unlike rejection sampling, SIR keeps all proposal samples in the first stage and uses weights to decide how often each appears in the resampled set.

## 15. Random Search Optimization

Monte Carlo ideas can also be used for optimization.

Suppose we want to minimize

\[
f(x)=(x-2)^2+\sin(5x),
\qquad x\in[-2,5].
\]

A random search samples many candidate points and keeps the best one.

In [ ]:
def objective(x):
    return (x - 2)**2 + np.sin(5*x)

N = 20_000
X = rng.uniform(-2, 5, size=N)
Y = objective(X)

best_idx = np.argmin(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

# Compare with numerical optimization
res = optimize.minimize_scalar(objective, bounds=(-2, 5), method="bounded")

print("Random search best x:", best_x)
print("Random search best f(x):", best_y)
print("Numerical optimizer x:", res.x)
print("Numerical optimizer f(x):", res.fun)

In [ ]:
x_grid = np.linspace(-2, 5, 800)
y_grid = objective(x_grid)

plt.figure(figsize=(7, 4))
plt.plot(x_grid, y_grid, label="objective")
plt.scatter([best_x], [best_y], label="random search best")
plt.scatter([res.x], [res.fun], marker="x", s=80, label="numerical optimizer")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Random Search Optimization")
plt.legend()
plt.show()

### Full Solution

Random search estimates the minimum by sampling candidates

\[
X_1,\ldots,X_N\sim U([-2,5])
\]

and selecting

\[
\hat x=\arg\min_{1\le i\le N} f(X_i).
\]

This method is simple and sometimes useful when derivatives are unavailable, but it can be inefficient in high dimensions.

## 16. Simulated Annealing

Simulated annealing is a stochastic optimization method.

At each step, propose a move. If it improves the objective, accept it.  
If it makes the objective worse, accept it with probability

\[
\exp\left(-\frac{\Delta}{T}\right),
\]

where $\Delta$ is the increase in objective value and $T$ is the temperature.

As $T$ decreases, the algorithm becomes less likely to accept worse moves.

In [ ]:
def simulated_annealing(f, x0, steps=8000, proposal_sd=0.4, T0=1.0, cooling=0.999):
    x = x0
    fx = f(x)
    T = T0

    path_x = [x]
    path_fx = [fx]

    for _ in range(steps):
        x_prop = x + rng.normal(0, proposal_sd)
        # reflect or reject if outside domain
        if x_prop < -2 or x_prop > 5:
            path_x.append(x)
            path_fx.append(fx)
            T *= cooling
            continue

        f_prop = f(x_prop)
        delta = f_prop - fx

        if delta <= 0 or rng.random() < np.exp(-delta / T):
            x, fx = x_prop, f_prop

        path_x.append(x)
        path_fx.append(fx)
        T *= cooling

    return np.array(path_x), np.array(path_fx)

path_x, path_fx = simulated_annealing(objective, x0=-1.5, steps=10_000, proposal_sd=0.35, T0=1.0, cooling=0.9993)

print("Final x:", path_x[-1])
print("Final f(x):", path_fx[-1])
print("Best x visited:", path_x[np.argmin(path_fx)])
print("Best f visited:", path_fx.min())

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(path_fx)
plt.xlabel("iteration")
plt.ylabel("f(x)")
plt.title("Simulated Annealing Objective Value")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(x_grid, y_grid, label="objective")
plt.scatter(path_x[::100], objective(path_x[::100]), s=8, alpha=0.4, label="SA path every 100 steps")
plt.scatter([path_x[np.argmin(path_fx)]], [path_fx.min()], marker="x", s=100, label="best visited")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Simulated Annealing Search Path")
plt.legend()
plt.show()

### Full Solution

If a proposed move decreases the objective, then $\Delta\le0$ and the move is accepted.

If it increases the objective, then $\Delta>0$ and the move is accepted with probability

\[
\exp(-\Delta/T).
\]

When $T$ is large, the algorithm explores widely.  
When $T$ is small, the algorithm behaves more like local search.

This helps avoid getting trapped too early in local minima.

# Practice Problems with Full Solutions

## Practice Problem 1 — Monte Carlo Integral

Estimate

\[
I=\int_0^1 \frac{1}{1+x^2}\,dx.
\]

Compare with the exact answer.

In [ ]:
N = 100_000
U = rng.uniform(0, 1, size=N)
values = 1/(1 + U**2)

estimate = values.mean()
se = values.std(ddof=1) / np.sqrt(N)

true_value = np.arctan(1) - np.arctan(0)

print("MC estimate:", estimate)
print("Estimated SE:", se)
print("Exact value pi/4:", true_value)
print("95% MC CI:", (estimate - 1.96*se, estimate + 1.96*se))

### Solution

Let $U\sim U(0,1)$. Then

\[
E\left[\frac{1}{1+U^2}\right]
=
\int_0^1 \frac{1}{1+x^2}\,dx.
\]

The exact answer is

\[
\int_0^1 \frac{1}{1+x^2}\,dx
=
\arctan(1)-\arctan(0)
=
\frac{\pi}{4}.
\]

The Monte Carlo estimator is

\[
\hat I=\frac1N\sum_{i=1}^N \frac{1}{1+U_i^2}.
\]

## Practice Problem 2 — Estimate an Expectation

Let $X\sim N(0,1)$. Estimate

\[
E[\cos(X)].
\]

Compare with the exact answer

\[
E[\cos(X)]=e^{-1/2}.
\]

In [ ]:
N = 200_000
X = rng.normal(0, 1, size=N)
values = np.cos(X)

estimate = values.mean()
se = values.std(ddof=1)/np.sqrt(N)
true_value = np.exp(-0.5)

print("MC estimate:", estimate)
print("Estimated SE:", se)
print("Exact e^{-1/2}:", true_value)
print("95% MC CI:", (estimate - 1.96*se, estimate + 1.96*se))

### Solution

The Monte Carlo estimator is

\[
\hat\theta_N=\frac1N\sum_{i=1}^N \cos(X_i),
\qquad X_i\sim N(0,1).
\]

The exact value follows from the characteristic function of a standard normal:

\[
E[e^{itX}]=e^{-t^2/2}.
\]

Taking the real part at $t=1$ gives

\[
E[\cos X]=e^{-1/2}.
\]

## Practice Problem 3 — Inverse Transform Sampling

Generate samples from the distribution with CDF

\[
F(x)=x^3,\qquad 0<x<1.
\]

Find the inverse transform and verify the density.

In [ ]:
N = 100_000
U = rng.uniform(0, 1, size=N)

# F(x)=x^3, so x=U^(1/3)
X = U**(1/3)

grid = np.linspace(0, 1, 400)
pdf = 3*grid**2

plt.figure(figsize=(7, 4))
plt.hist(X, bins=70, density=True, alpha=0.7, label="samples")
plt.plot(grid, pdf, label=r"$f(x)=3x^2$")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Inverse Transform Sampling with F(x)=x^3")
plt.legend()
plt.show()

### Solution

Set

\[
U=F(X)=X^3.
\]

Solving gives

\[
X=U^{1/3}.
\]

The density is

\[
f(x)=F'(x)=3x^2,\qquad 0<x<1.
\]

## Practice Problem 4 — Box--Muller

Use the Box--Muller method to generate $10{,}000$ standard normal samples.  
Compute the sample mean and sample variance.

In [ ]:
N = 10_000
U1 = rng.uniform(0, 1, size=N//2)
U2 = rng.uniform(0, 1, size=N//2)

R = np.sqrt(-2*np.log(U1))
Theta = 2*np.pi*U2

Z1 = R*np.cos(Theta)
Z2 = R*np.sin(Theta)

Z = np.concatenate([Z1, Z2])

print("Sample mean:", Z.mean())
print("Sample variance:", Z.var(ddof=1))

### Solution

The Box--Muller formulas are

\[
Z_1=\sqrt{-2\log U_1}\cos(2\pi U_2),
\]

\[
Z_2=\sqrt{-2\log U_1}\sin(2\pi U_2).
\]

They produce independent $N(0,1)$ random variables. Therefore, for a large sample, the sample mean should be close to $0$ and the sample variance should be close to $1$.

## Practice Problem 5 — Rejection Sampling

Use rejection sampling to sample from the density

\[
p(x)=2x,\qquad 0<x<1,
\]

using a Uniform$(0,1)$ proposal.

In [ ]:
# Target p(x)=2x on (0,1), proposal q(x)=1.
# Need k=2.

N = 50_000
accepted = []
total = 0
k = 2

while len(accepted) < N:
    m = max(1000, 2*(N-len(accepted)))
    z = rng.uniform(0, 1, size=m)
    u = rng.uniform(0, 1, size=m)
    accept = u <= (2*z) / k
    accepted.extend(z[accept].tolist())
    total += m

accepted = np.array(accepted[:N])
accept_rate = N/total

grid = np.linspace(0, 1, 400)

plt.figure(figsize=(7, 4))
plt.hist(accepted, bins=70, density=True, alpha=0.7, label="samples")
plt.plot(grid, 2*grid, label=r"$p(x)=2x$")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Rejection Sampling Practice")
plt.legend()
plt.show()

print("Acceptance rate:", accept_rate)
print("Expected acceptance rate:", 1/k)
print("Sample mean:", accepted.mean())
print("Theoretical mean:", 2/3)

### Solution

The proposal is

\[
q(x)=1,\qquad 0<x<1.
\]

The target satisfies

\[
p(x)=2x\le 2=kq(x).
\]

Therefore, use $k=2$.

Accept a proposal $Z$ if

\[
U\le \frac{p(Z)}{kq(Z)}
=
\frac{2Z}{2}
=
Z.
\]

The accepted samples follow density $p(x)=2x$. The expected acceptance rate is

\[
\frac1k=\frac12.
\]

## Practice Problem 6 — Importance Sampling for a Rare Event

Estimate

\[
P(Z>4)
\]

where $Z\sim N(0,1)$ using importance sampling with proposal $q=N(4,1)$.

In [ ]:
N = 100_000
proposal_mean = 4

X = rng.normal(proposal_mean, 1, size=N)
weights = stats.norm.pdf(X, 0, 1) / stats.norm.pdf(X, proposal_mean, 1)

values = (X > 4).astype(float) * weights
estimate = values.mean()
se = values.std(ddof=1)/np.sqrt(N)

true_value = 1 - stats.norm.cdf(4)

print("IS estimate:", estimate)
print("Estimated SE:", se)
print("True value:", true_value)
print("95% MC CI:", (estimate - 1.96*se, estimate + 1.96*se))

### Solution

The target probability is

\[
P(Z>4)=E_p[1\{Z>4\}].
\]

Use proposal $q=N(4,1)$. Then

\[
P(Z>4)
=
E_q\left[
1\{X>4\}\frac{p(X)}{q(X)}
\right].
\]

The estimator is

\[
\hat p
=
\frac1N\sum_{i=1}^N
1\{X_i>4\}
\frac{\phi(X_i)}{\phi(X_i-4)},
\qquad X_i\sim N(4,1).
\]

## Practice Problem 7 — Self-Normalized Importance Sampling

Suppose the unnormalized target is

\[
\tilde p(x)=e^{-|x|}.
\]

This is proportional to a Laplace distribution centered at $0$. Use proposal $q=N(0,2^2)$ to estimate

\[
E_p[X^2].
\]

The exact value for Laplace$(0,1)$ is

\[
E[X^2]=2.
\]

In [ ]:
N = 200_000

X = rng.normal(0, 2, size=N)

p_tilde_laplace = np.exp(-np.abs(X))
q = stats.norm.pdf(X, 0, 2)

raw_w = p_tilde_laplace / q
w = raw_w / raw_w.sum()

estimate = np.sum(w * X**2)
ess = 1/np.sum(w**2)

print("Self-normalized IS estimate E[X^2]:", estimate)
print("Exact value:", 2.0)
print("Effective sample size:", ess)

### Solution

The self-normalized importance sampling estimator is

\[
\hat\theta
=
\sum_{i=1}^N W_i X_i^2,
\]

where

\[
W_i
=
\frac{\tilde p(X_i)/q(X_i)}
{\sum_{m=1}^N \tilde p(X_m)/q(X_m)}.
\]

Here

\[
\tilde p(x)=e^{-|x|},
\]

which normalizes to the Laplace$(0,1)$ density

\[
p(x)=\frac12e^{-|x|}.
\]

For Laplace$(0,1)$,

\[
E[X^2]=2.
\]

## Practice Problem 8 — Monte Carlo Error and Confidence Interval

Suppose $U_i\sim U(0,1)$ and

\[
\hat\theta=\frac1N\sum_{i=1}^N \sqrt{U_i}.
\]

Estimate

\[
\theta=E[\sqrt U].
\]

Construct an approximate 95% Monte Carlo confidence interval and compare with the exact answer.

In [ ]:
N = 50_000
U = rng.uniform(0, 1, size=N)
values = np.sqrt(U)

estimate = values.mean()
se = values.std(ddof=1)/np.sqrt(N)
ci = (estimate - 1.96*se, estimate + 1.96*se)

true_value = 2/3

print("MC estimate:", estimate)
print("Estimated SE:", se)
print("95% MC CI:", ci)
print("Exact value:", true_value)

### Solution

Since $U\sim U(0,1)$,

\[
E[\sqrt U]
=
\int_0^1 \sqrt u\,du
=
\left[\frac{2}{3}u^{3/2}\right]_0^1
=
\frac23.
\]

The Monte Carlo estimator is

\[
\hat\theta=\frac1N\sum_{i=1}^N\sqrt{U_i}.
\]

Estimate its standard error by

\[
\widehat{\operatorname{SE}}(\hat\theta)
=
\frac{s}{\sqrt N},
\]

where $s$ is the sample standard deviation of $\sqrt{U_i}$.

An approximate 95% Monte Carlo confidence interval is

\[
\hat\theta\pm1.96\,\widehat{\operatorname{SE}}.
\]

# Summary

In this lab, we used computation to study the main ideas of Section 10:

- Monte Carlo estimates expectations with sample averages.
- The Law of Large Numbers explains convergence.
- Standard errors decrease at rate $1/\sqrt N$.
- Monte Carlo integration converts integrals into expectations.
- Inverse-transform sampling converts uniforms into target random variables.
- Box--Muller converts uniforms into normal random variables.
- Rejection sampling draws from complex targets using a proposal and envelope.
- Importance sampling improves efficiency by sampling from a better distribution.
- Self-normalized importance sampling and SIR handle unnormalized targets.
- Random search and simulated annealing apply Monte Carlo ideas to optimization.

The central message is:

\[
\boxed{
\text{Monte Carlo uses randomness as a computational tool.}
}
\]